In [1]:
from __future__ import annotations

import os
import csv
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import glob


@dataclass(frozen=True)
class ResultRow:
    cv_id: int
    seed: int
    val_metric: str
    val_score: float
    test_acc: float
    test_macro_f1: float
    test_top5: float
    best_trial: int
    timestamp: str
    run_dir: str  # cv-*_seed-* directory name


def _safe_float(x: Any) -> float:
    try:
        return float(x)
    except Exception:
        return float("nan")


def _safe_int(x: Any) -> int:
    try:
        return int(float(x))
    except Exception:
        return -1


def _mean(xs: List[float]) -> float:
    xs = [x for x in xs if x == x]  # drop NaN
    return sum(xs) / len(xs) if xs else float("nan")


def _std(xs: List[float]) -> float:
    xs = [x for x in xs if x == x]
    if len(xs) <= 1:
        return float("nan")
    m = _mean(xs)
    v = sum((x - m) ** 2 for x in xs) / (len(xs) - 1)
    return v ** 0.5


def _min(xs: List[float]) -> float:
    xs = [x for x in xs if x == x]
    return min(xs) if xs else float("nan")


def _max(xs: List[float]) -> float:
    xs = [x for x in xs if x == x]
    return max(xs) if xs else float("nan")


def _read_metrics_csv(metrics_path: Path) -> Optional[Dict[str, Any]]:
    if not metrics_path.exists():
        return None
    with metrics_path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    if not rows:
        return None
    return rows[0]


def _parse_cv_seed_from_dirname(name: str) -> Tuple[int, int]:
    # expected: cv-0_seed-12
    m = re.match(r"cv-(\d+)_seed-(\d+)", name)
    if not m:
        return -1, -1
    return int(m.group(1)), int(m.group(2))


def load_results(save_root: str | Path, include_best_params: bool = False) -> Tuple[List[ResultRow], Optional[List[Dict[str, Any]]]]:
    """
    save_root/
      cv-0_seed-0/metrics.csv
      cv-0_seed-0/best_param.json (optional)
      ...

    Returns:
      rows: list of ResultRow
      best_params_list: list of dict (if include_best_params=True else None)
    """
    save_root = Path(save_root)
    if not save_root.exists():
        raise FileNotFoundError(f"save_root not found: {save_root}")

    rows: List[ResultRow] = []
    best_params_list: List[Dict[str, Any]] = []

    subdirs = sorted([p for p in save_root.iterdir() if p.is_dir()])

    for d in subdirs:
        cv_id, seed = _parse_cv_seed_from_dirname(d.name)
        metrics_path = d / "metrics.csv"
        r = _read_metrics_csv(metrics_path)
        if r is None:
            # metrics.csv が無い/壊れている場合はスキップ
            continue

        row = ResultRow(
            cv_id=cv_id if cv_id != -1 else _safe_int(r.get("cv_id", -1)),
            seed=seed if seed != -1 else _safe_int(r.get("seed", -1)),
            val_metric=str(r.get("val_metric", "")),
            val_score=_safe_float(r.get("val_score", "nan")),
            test_acc=_safe_float(r.get("test_acc", "nan")),
            test_macro_f1=_safe_float(r.get("test_macro_f1", "nan")),
            test_top5=_safe_float(r.get("test_top5", "nan")),
            best_trial=_safe_int(r.get("best_trial", -1)),
            timestamp=str(r.get("timestamp", "")),
            run_dir=d.name,
        )
        rows.append(row)

        if include_best_params:
            bp = d / "best_param.json"
            if bp.exists():
                try:
                    best_params_list.append(json.loads(bp.read_text(encoding="utf-8")))
                except Exception:
                    best_params_list.append({"__error__": f"failed to read {bp}"})

    return rows, (best_params_list if include_best_params else None)


def print_results(save_root: str | Path, include_best_params: bool = False, include_cv: bool = False) -> None:
    """
    保存先を入力すると結果を表示する関数である。
    """
    rows, best_params_list = load_results(save_root, include_best_params=include_best_params)
    if not rows:
        print(f"[WARN] No results found under: {Path(save_root).resolve()}", flush=True)
        return

    # 全体集計
    accs = [r.test_acc for r in rows]
    f1s = [r.test_macro_f1 for r in rows]
    top5s = [r.test_top5 for r in rows]
    vals = [r.val_score for r in rows]

    print("=" * 72, flush=True)
    print(f"[RESULTS] root = {Path(save_root).resolve()}", flush=True)
    print(f"[RESULTS] runs = {len(rows)} (cv x seed)", flush=True)
    print("-" * 72, flush=True)
    print(
        "TEST  acc     : mean={:.4f} std={:.4f} min={:.4f} max={:.4f}".format(
            _mean(accs), _std(accs), _min(accs), _max(accs)
        ),
        flush=True,
    )
    print(
        "TEST  macro_f1: mean={:.4f} std={:.4f} min={:.4f} max={:.4f}".format(
            _mean(f1s), _std(f1s), _min(f1s), _max(f1s)
        ),
        flush=True,
    )
    print(
        "TEST  top5    : mean={:.4f} std={:.4f} min={:.4f} max={:.4f}".format(
            _mean(top5s), _std(top5s), _min(top5s), _max(top5s)
        ),
        flush=True,
    )
    print(
        "VAL({}) score : mean={:.4f} std={:.4f}".format(
            rows[0].val_metric, _mean(vals), _std(vals)
        ),
        flush=True,
    )

    # foldごとの平均（seed平均）
    by_cv: Dict[int, List[ResultRow]] = {}
    for r in rows:
        by_cv.setdefault(r.cv_id, []).append(r)

    if include_cv:
        print("-" * 72, flush=True)
        print("[BY CV] (seed average per fold)", flush=True)
        for cv_id in sorted(by_cv.keys()):
            rs = by_cv[cv_id]
            a = _mean([x.test_acc for x in rs])
            f = _mean([x.test_macro_f1 for x in rs])
            t = _mean([x.test_top5 for x in rs])
            print(f"  cv={cv_id:2d} | n={len(rs):2d} | acc={a:.4f}  f1={f:.4f}  top5={t:.4f}", flush=True)

        # 代表としてトップ数件の個別結果
        print("-" * 72, flush=True)
        print("[TOP RUNS] by test_acc (top 10)", flush=True)
        rows_sorted = sorted(rows, key=lambda r: (r.test_acc if r.test_acc == r.test_acc else -1.0), reverse=True)
        for i, r in enumerate(rows_sorted[:10]):
            print(
                f"  #{i+1:2d} {r.run_dir:14s} | acc={r.test_acc:.4f} f1={r.test_macro_f1:.4f} top5={r.test_top5:.4f} | val={r.val_score:.4f}",
                flush=True,
            )

    if include_best_params and best_params_list is not None:
        # best paramを軽く眺める（完全表示だと長くなるので、上位の試行だけ表示）
        print("-" * 72, flush=True)
        print("[BEST PARAMS] (first 3 entries)", flush=True)
        for i, bp in enumerate(best_params_list[:3]):
            print(f"  --- {i+1} ---", flush=True)
            # よく見る所だけ出す
            trial_params = bp.get("best_trial_params", {})
            resolved = bp.get("resolved_hyperparams", {})
            print(f"  best_value: {bp.get('best_value')}", flush=True)
            print(f"  best_trial_params: {trial_params}", flush=True)
            print(f"  resolved_hyperparams: {resolved}", flush=True)

    print("=" * 72, flush=True)

In [2]:
import os
import glob
import json
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

ROOT_PATH = "/home/nakanishi/WORKSPACE/bidirectional_2D_reservoir_computing/results"


def match_run_dirs(dataset: str, model: str, units: int) -> List[str]:
    """
    例: .../results/**/2026-02-02_12-05-05_mnist_esn_units-128
    """
    pattern = os.path.join(ROOT_PATH, "**", f"*_{dataset}_{model}_units-{units}")
    return sorted(glob.glob(pattern, recursive=True))


def _read_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_fold_result(fold_dir: Path) -> Dict[str, Any]:
    """
    fold_dir 例: <run_dir>/cv-0_seed-0
    ここから info.json / best_param.json / metrics.csv を読み出して統合する。
    """
    info_path = fold_dir / "info.json"
    best_param_path = fold_dir / "best_param.json"
    metrics_path = fold_dir / "metrics.csv"

    if not info_path.exists():
        raise FileNotFoundError(f"Missing info.json: {info_path}")
    if not best_param_path.exists():
        raise FileNotFoundError(f"Missing best_param.json: {best_param_path}")
    if not metrics_path.exists():
        raise FileNotFoundError(f"Missing metrics.csv: {metrics_path}")

    info = _read_json(info_path)
    best_param = _read_json(best_param_path)

    # metrics.csv は1行だけ
    mdf = pd.read_csv(metrics_path)
    if len(mdf) != 1:
        raise ValueError(f"metrics.csv should have exactly 1 row, got {len(mdf)}: {metrics_path}")
    metrics = mdf.iloc[0].to_dict()

    # best params（あなたが例で挙げたのは best_trial_params）
    best_trial_params = best_param.get("best_trial_params", {}) or {}
    resolved_hp = best_param.get("resolved_hyperparams", {}) or {}

    # connectivity/leaky/spectral_radius/beta は resolved_hyperparams 側に入っているはず（固定でも）
    # best_trial_params は「tuneしてないと入らない」ので、両方から拾う（resolved優先）
    def pick_hp(name: str) -> Optional[float]:
        if name in resolved_hp:
            return resolved_hp.get(name)
        return best_trial_params.get(name)

    return {
        "cv": int(info.get("cv_id")),
        "seed": int(info.get("seed")),
        "acc": float(metrics.get("test_acc")),
        "f1": float(metrics.get("test_macro_f1")),
        "top5": float(metrics.get("test_top5")),
        "connectivity": pick_hp("connectivity"),
        "leaky": pick_hp("leaky"),
        "spectral_radius": pick_hp("spectral_radius"),
        "beta": pick_hp("beta"),
        # 参考（必要なら）
        "val_metric": metrics.get("val_metric"),
        "val_score": metrics.get("val_score"),
        "best_trial": metrics.get("best_trial"),
        "elapsed_sec": info.get("elapsed_sec"),
        "timestamp": metrics.get("timestamp"),
    }


def collect_results_to_csv(
    datasets: List[str],
    models: List[str],
    units_list: List[int],
    output_csv: str,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for dataset in datasets:
        for units in units_list:
            for model in models:
                run_dirs = match_run_dirs(dataset, model, units)
                if len(run_dirs) == 0:
                    print(f"[WARN] run_dir not found: dataset={dataset}, model={model}, units={units}")
                    continue

                for run_dir in run_dirs:
                    run_path = Path(run_dir)

                    # あなたの保存形式：run_dir/cv-<id>_seed-<seed>/
                    fold_dirs = sorted([p for p in run_path.iterdir() if p.is_dir() and p.name.startswith("cv-")])

                    if len(fold_dirs) == 0:
                        print(f"[WARN] No fold dirs under: {run_dir}")
                        continue

                    for fold_dir in fold_dirs:
                        try:
                            r = load_fold_result(fold_dir)
                            row = {
                                "dataset": dataset,
                                "model": model,
                                "units": units,
                                "cv": r["cv"],
                                "seed": r["seed"],
                                "acc": r["acc"],
                                "f1": r["f1"],
                                "top5": r["top5"],
                                "connectivity": r["connectivity"],
                                "leaky": r["leaky"],
                                "spectral_radius": r["spectral_radius"],
                                "beta": r["beta"],
                                # いらなければ消してOK
                                "val_metric": r["val_metric"],
                                "val_score": r["val_score"],
                                "best_trial": r["best_trial"],
                                "elapsed_sec": r["elapsed_sec"],
                                "timestamp": r["timestamp"],
                                "run_dir": str(run_path),
                                "fold_dir": str(fold_dir),
                            }
                            rows.append(row)
                        except Exception as e:
                            print(f"[WARN] Failed fold: {fold_dir}\n  -> {repr(e)}")

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["dataset", "units", "model", "seed", "cv"], kind="stable")

    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"[OK] Saved CSV: {output_csv}  (rows={len(df)})")
    return df


collect_results_to_csv(
    datasets=["mnist", "cifar_10", "stl_10"],
    models=["esn", "bi_esn", "bi_esn2d"],
    units_list=[128, 256, 512, 1024],
    output_csv=os.path.join(
        "/home/nakanishi/WORKSPACE/bidirectional_2D_reservoir_computing/results/nm2026_out", "results.csv"
    ),
)

[OK] Saved CSV: /home/nakanishi/WORKSPACE/bidirectional_2D_reservoir_computing/results/nm2026_out/results.csv  (rows=540)


,dataset,model,units,cv,seed,acc,f1,top5,connectivity,leaky,spectral_radius,beta,val_metric,val_score,best_trial,elapsed_sec,timestamp,run_dir,fold_dir
195,cifar_10,bi_esn,128,0,0,0.415500,0.405245,0.889300,0.120877,0.641700,0.978746,0.000032,acc,0.4192,25,1910.368133,2026-02-03T14:16:20.757026,/home/nakanishi/WORKSPACE/bidirectional_2D_res...,/home/nakanishi/WORKSPACE/bidirectional_2D_res...
198,cifar_10,bi_esn,128,1,0,0.415100,0.404296,0.885400,0.088254,0.684363,0.902287,0.000016,acc,0.4148,6,1971.469440,2026-02-03T14:49:13.520361,/home/nakanishi/WORKSPACE/bidirectional_2D_res...,/home/nakanishi/WORKSPACE/bidirectional_2D_res...
201,cifar_10,bi_esn,128,2,0,0.421400,0.412209,0.884200,0.377578,0.648718,0.898761,0.000062,acc,0.4216,8,1932.133757,2026-02-03T15:21:25.993415,/home/nakanishi/WORKSPACE/bidirectional_2D_res...,/home/nakanishi/WORKSPACE/bidirectional_2D_res...
204,cifar_10,bi_esn,128,3,0,0.406200,0.396803,0.873500,0.164736,0.533567,0.920706,0.000022,acc,0.4208,11,1926.332934,2026-02-03T15:53:32.647449,/home/nakanishi/WORKSPACE/bidirectional_2D_res...,/home/nakanishi/WORKSPACE/bidirectional_2D_res...
207,cifar_10,bi_esn,128,4,0,0.411200,0.401368,0.883200,0.283361,0.678844,0.950106,0.000427,acc,0.4221,24,1949.415563,2026-02-03T16:26:03.283494,/home/nakanishi/WORKSPACE/bidirectional_2D_res...,/home/nakanishi/WORKSPACE/bidirectional_2D_res...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,stl_10,esn,1024,0,2,0.344750,0.341073,0.794250,0.381251,0.830822,0.981739,0.000010,acc,0.3690,10,534.680795,2026-02-06T15:02:51.646500,/home/nakanishi/WORKSPACE/bidirectional_2D_res...,/home/nakanishi/WORKSPACE/bidirectional_2D_res...
500,stl_10,esn,1024,1,2,0.345250,0.340647,0.797625,0.187877,0.741433,0.923330,0.000116,acc,0.3700,26,544.928961,2026-02-06T15:11:56.772251,/home/nakanishi/WORKSPACE/bidirectional_2D_res...,/home/nakanishi/WORKSPACE/bidirectional_2D_res...
503,stl_10,esn,1024,2,2,0.347125,0.343011,0.794750,0.533828,0.501599,0.951104,0.000018,acc,0.3720,23,534.249425,2026-02-06T15:20:51.221745,/home/nakanishi/WORKSPACE/bidirectional_2D_res...,/home/nakanishi/WORKSPACE/bidirectional_2D_res...
506,stl_10,esn,1024,3,2,0.343875,0.340893,0.788500,0.126133,0.737552,0.949171,0.000020,acc,0.3540,28,534.371092,2026-02-06T15:29:45.780068,/home/nakanishi/WORKSPACE/bidirectional_2D_res...,/home/nakanishi/WORKSPACE/bidirectional_2D_res...


---

In [3]:
ROOT_PATH = "/home/nakanishi/WORKSPACE/bidirectional_2D_reservoir_computing/results/nm2026"


def match_dir(dataset: str, model: str, units: int) -> List[str]:
    pattern = os.path.join(ROOT_PATH, f"*_{dataset}_{model}_units-{units}")
    matched_dirs = glob.glob(pattern)
    return matched_dirs[0]


flag = False

for dataset in ["mnist", "cifar_10", "stl_10"]:
    for units in [128, 256, 512, 1024]:
        print_results(match_dir(dataset, "esn", units), include_best_params=flag, include_cv=flag)
        print_results(match_dir(dataset, "bi_esn", units), include_best_params=flag, include_cv=flag)
        print_results(match_dir(dataset, "bi_esn2d", units), include_best_params=flag, include_cv=flag)

[RESULTS] root = /home/nakanishi/WORKSPACE/bidirectional_2D_reservoir_computing/results/nm2026/2026-02-02_12-05-05_mnist_esn_units-128
[RESULTS] runs = 15 (cv x seed)
------------------------------------------------------------------------
TEST  acc     : mean=0.7839 std=0.0226 min=0.7148 max=0.8115
TEST  macro_f1: mean=0.7791 std=0.0233 min=0.7080 max=0.8068
TEST  top5    : mean=0.9772 std=0.0045 min=0.9636 max=0.9827
VAL(acc) score : mean=0.8028 std=0.0067
[RESULTS] root = /home/nakanishi/WORKSPACE/bidirectional_2D_reservoir_computing/results/nm2026/2026-02-02_16-26-23_mnist_bi_esn_units-128
[RESULTS] runs = 15 (cv x seed)
------------------------------------------------------------------------
TEST  acc     : mean=0.8008 std=0.0088 min=0.7871 max=0.8181
TEST  macro_f1: mean=0.7971 std=0.0091 min=0.7831 max=0.8147
TEST  top5    : mean=0.9805 std=0.0021 min=0.9762 max=0.9848
VAL(acc) score : mean=0.8095 std=0.0101
[RESULTS] root = /home/nakanishi/WORKSPACE/bidirectional_2D_reservoir_c